$$ \text{Newton's method for finding extrema} $$
$$ \text{Single-variable case:} $$
$$ f(x) \approx f(x_0) + f'(x_0)(x - x_0) + \frac{1}{2} f''(x_0)(x - x_0)^2 $$
$$ \frac{df}{dx} = 0 $$
$$ \frac{df}{dx} = f'(x_0) + f''(x_0)(x - x_0) $$
$$ x = x_0 - \frac{f'(x_0)}{f''(x_0)} $$
$$ \\[2em] $$
$$ \text{Example 1:} $$
$$ f(x) = x^2 $$
$$ f'(x) = 2x $$
$$ f''(x) = 2 $$
$$ x_{\text{new}} = x - \frac{2x}{2} = 0 $$
$$ \\[2em] $$
$$ \text{Example 2:} $$
$$ f(x) = 2x^2 + 3x - 1 $$
$$ f'(x) = 4x + 3 $$
$$ f''(x) = 4 $$
$$ x_{\text{new}} = x - \frac{4x + 3}{4} = -\frac{3}{4} $$
$$ \\[2em] $$

In [ ]:
from math import sin

import torch as tr
import torch.autograd as autograd

import import_ipynb
from common import check_eq, assert_eq # type: ignore


def newton_1d(f: callable, x0: float, epochs=20, atol=0.01) -> float:
    """
    Perform Newton's method to find a local minimum of a single-variable function `f` starting from an initial guess `x0`.
    
    Args:
        f: A single-variable function that takes a Tensor input and returns a scalar Tensor output.
        x0: A float representing the initial guess for the minimum.
        epochs: The maximum number of iterations to perform.
        atol: The absolute tolerance for convergence. The method will stop if the norm of the change in `x` is less than `atol`.
    
    Returns:
        A float representing the estimated location of the minimum of `f`.
    """
    
    x = tr.tensor(x0, dtype=tr.float32, requires_grad=True)
    check_eq(x.shape, [])

    for _ in range(epochs):
        y = f(x)
        assert_eq(y.shape, ())

        (dfdx, ) = autograd.grad(y, x, create_graph=True)
        check_eq(dfdx.shape, [])

        (dfdx2, ) = autograd.grad(dfdx, x, create_graph=False)
        check_eq(dfdx.shape, [])

        if check_eq(dfdx, 0.0, atol=1e-6):
            break

        x_new = x - dfdx / dfdx2
        check_eq(x_new.shape, [])

        if check_eq(x, x_new, atol=atol):
            return x_new.item()

        x = x_new.detach().requires_grad_()

    return x.item()


def pow2(x: tr.Tensor) -> tr.Tensor:
    return x ** 2


def test_newton_1d() -> None:
    # it works with tensor functions
    assert_eq(newton_1d(pow2, x0=1.0), 0.0, atol=0.01)

    # it works with lambdas due to `lambda` → `tensor function` conversion on the fly
    assert_eq(newton_1d(lambda x: 2*x**2 + 3*x - 1, x0=1.0), -0.75, atol=0.01)

    # it does not work with non-tensor functions, 
    # because they do not have a graph to compute the derivatives
    # assert_eq(newton_1d(lambda x: sin(x), x0=1.0), -0.75, atol=0.01)


if __name__ == "__main__":
    test_newton_1d()